In [1]:
# =====================================
# IMPORTS
# =====================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [2]:
# =====================================
# LOAD PREPROCESSED DATA
# =====================================

model_data = pd.read_pickle("../data/preprocessed_model_data.pkl")

print("Loaded preprocessed data:")
print(model_data.shape)
print(model_data.columns.tolist())


Loaded preprocessed data:
(45924, 24)
['Confirmation Year', 'Handler Region', 'Product Group', 'Product Type', 'Product Type Code', 'Industry Name', 'Industry Code', 'Service Line Code', 'Service Line Name', 'Service Detail', 'Service Program', 'Service Catalog Category', 'Service Catalog Item Number', 'Service Catalog Segment', 'Service Catalog Sub Category', 'CCN', 'Ship to Customer Region', 'Has Test Task Flag', 'Ship to Account Number', 'Flex Standards', 'Standard Count', 'Flex Project Count', 'Test Count', 'Log_Eng_Hours']


In [ ]:
# =====================================
# CELL 3 - CREATE X AND y WITH CONTROLLED ONE-HOT ENCODING
# =====================================

#695 features overall

import numpy as np
import pandas as pd

# -----------------------------
# 1. Define X and y
# -----------------------------

X = model_data.drop(columns=["Log_Eng_Hours"]).copy()
y = model_data["Log_Eng_Hours"].copy()

# Treat account number as a category, not a true number
if "Ship to Account Number" in X.columns:
    X["Ship to Account Number"] = X["Ship to Account Number"].astype("string")
print(X["Ship to Account Number"].dtype)

# -----------------------------
# 2. Identify categorical columns
# -----------------------------

cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Categorical columns before grouping:")
for col in cat_cols:
    print(f"{col}: {X[col].nunique()} unique values")

# -----------------------------
# 3. Limit category explosion
# Keep top categories, group the rest as OTHER
# -----------------------------

MAX_CATEGORIES_PER_COLUMN = 50

X_controlled = X.copy()

for col in cat_cols:
    X_controlled[col] = X_controlled[col].astype("string").fillna("UNKNOWN").str.strip()
    X_controlled[col] = X_controlled[col].replace({"": "UNKNOWN", "nan": "UNKNOWN", "None": "UNKNOWN"})
    
    top_categories = X_controlled[col].value_counts().head(MAX_CATEGORIES_PER_COLUMN).index
    
    X_controlled[col] = np.where(
        X_controlled[col].isin(top_categories),
        X_controlled[col],
        "OTHER"
    )

# -----------------------------
# 4. One-hot encode AFTER grouping
# -----------------------------

X_encoded = pd.get_dummies(
    X_controlled,
    columns=cat_cols,
    drop_first=True,
    dtype=int
)

# -----------------------------
# 5. Clean column names
# -----------------------------

X_encoded.columns = (
    X_encoded.columns
    .astype(str)
    .str.replace("[", "_", regex=False)
    .str.replace("]", "_", regex=False)
    .str.replace("<", "_", regex=False)
)

# -----------------------------
# 6. Confirm results
# -----------------------------

print("\nOriginal X shape:", X.shape)
print("Controlled X shape:", X_controlled.shape)
print("Final X_encoded shape:", X_encoded.shape)
print("y shape:", y.shape)

print("\nFinal encoded feature count:", X_encoded.shape[1])



string
Categorical columns before grouping:
Confirmation Year: 3 unique values
Handler Region: 3 unique values
Product Group: 218 unique values
Product Type: 233 unique values
Product Type Code: 10 unique values
Industry Name: 31 unique values
Industry Code: 31 unique values
Service Line Code: 167 unique values
Service Line Name: 167 unique values
Service Detail: 386 unique values
Service Program: 120 unique values
Service Catalog Category: 73 unique values
Service Catalog Item Number: 404 unique values
Service Catalog Segment: 22 unique values
Service Catalog Sub Category: 348 unique values
CCN: 209 unique values
Ship to Customer Region: 3 unique values
Has Test Task Flag: 2 unique values
Ship to Account Number: 6157 unique values
Flex Standards: 201 unique values

Original X shape: (45924, 23)
Controlled X shape: (45924, 23)
Final X_encoded shape: (45924, 695)
y shape: (45924,)

Final encoded feature count: 695


In [4]:
# =====================================
# TRAIN / TEST SPLIT
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.30,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (32146, 695)
X_test: (13778, 695)
y_train: (32146,)
y_test: (13778,)


In [5]:
#Evaluation Function

def evaluate_model(model, X_test, y_test):
    y_pred_log = model.predict(X_test)

    y_pred = np.expm1(y_pred_log)
    y_true = np.expm1(y_test)

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(y_true, y_pred)
    )

    r2 = r2_score(y_test, y_pred_log)

    return mae, rmse, r2


In [6]:
# =====================================
# TRAIN MODELS
# =====================================

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.001, max_iter=10000),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        objective="reg:squarederror"
    )
}

results_list = []

for name, model in models.items():
    print(f"Training {name}...")

    model.fit(X_train, y_train)

    mae, rmse, r2 = evaluate_model(model, X_test, y_test)

    results_list.append({
        "Model": name,
        "MAE_hours": mae,
        "RMSE_hours": rmse,
        "R2_log_scale": r2
    })

results = pd.DataFrame(results_list).sort_values("RMSE_hours")

results


Training Linear Regression...
Training Ridge Regression...
Training Lasso Regression...
Training Random Forest...
Training XGBoost...


,Model,MAE_hours,RMSE_hours,R2_log_scale
3,Random Forest,6.263889,15.078298,0.509924
4,XGBoost,6.433244,15.969042,0.497671
0,Linear Regression,7.003407,16.851106,0.411098
1,Ridge Regression,7.000508,16.896790,0.412923
2,Lasso Regression,7.251724,17.249796,0.374737


In [7]:
# =====================================
# RANDOM FOREST ONLY - 5 FOLD CV (MAE)
# =====================================

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, make_scorer
import numpy as np

# 5-fold setup
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# MAE scorer on real hours (not log scale)
def mae_real_hours(y_true_log, y_pred_log):
    y_true_hours = np.expm1(y_true_log)
    y_pred_hours = np.expm1(y_pred_log)

    return mean_absolute_error(
        y_true_hours,
        y_pred_hours
    )

mae_scorer = make_scorer(
    mae_real_hours,
    greater_is_better=False
)

# Get Random Forest from your models dictionary
rf = models["Random Forest"]

print("Running 5-Fold CV for Random Forest...")

scores = cross_val_score(
    rf,
    X_encoded,
    y,
    cv=kf,
    scoring=mae_scorer,
    n_jobs=1
)

mae_scores = -scores

print("\nFold MAEs:")
for i, score in enumerate(mae_scores, start=1):
    print(f"Fold {i}: {score:.4f}")

print("\nAverage MAE:", round(mae_scores.mean(), 4))
print("MAE Std Dev:", round(mae_scores.std(), 4))



Running 5-Fold CV for Random Forest...

Fold MAEs:
Fold 1: 6.2468
Fold 2: 6.3167
Fold 3: 6.2693
Fold 4: 6.1591
Fold 5: 6.1456

Average MAE: 6.2275
MAE Std Dev: 0.0655


In [8]:
# ============================================================
# DEVIATION BUCKET SUMMARY TABLES
# 100% Data, 70% Training Data, 30% Test Data
# ============================================================

import pandas as pd
import numpy as np

# Use best model
best_model = models["Random Forest"]

# Helper function to create bucket summary
def create_deviation_summary(X_data, y_data, label):
    # Predict log hours
    y_pred_log = best_model.predict(X_data)

    # Convert log hours back to real hours
    actual_hours = np.expm1(y_data)
    predicted_hours = np.expm1(y_pred_log)

    # Create results table
    temp = pd.DataFrame({
        "Actual Human": actual_hours,
        "Model Estimate": predicted_hours
    })

    # Difference between actual and predicted
    temp["Deviation"] = abs(temp["Actual Human"] - temp["Model Estimate"])

    # Buckets
    temp["Deviation Buckets"] = pd.cut(
        temp["Deviation"],
        bins=[-0.001, 1, 2, 3, 7, np.inf],
        labels=["<1 hour", "1-2 hours", "2-3 hours", "3-7 hours", ">7 hours"]
    )

    # Summary
    summary = temp.groupby("Deviation Buckets").agg(
        **{
            "# Projects": ("Deviation", "count"),
            "Actual Human": ("Actual Human", "mean"),
            "Model Estimate Median": ("Model Estimate", "median")
        }
    ).reset_index()

    # Percent of projects
    summary["% of Projects"] = (
        summary["# Projects"] / summary["# Projects"].sum() * 100
    )

    # Reorder columns
    summary = summary[
        ["Deviation Buckets", "# Projects", "% of Projects", "Actual Human", "Model Estimate Median"]
    ]

    # Round
    summary["% of Projects"] = summary["% of Projects"].round(0).astype(int).astype(str) + "%"
    summary["Actual Human"] = summary["Actual Human"].round(2)
    summary["Model Estimate Median"] = summary["Model Estimate Median"].round(2)

    print("\n" + "="*60)
    print(label)
    print("="*60)
    display(summary)

    return summary


# 100% of data
summary_100 = create_deviation_summary(
    X_encoded,
    y,
    "100% of Data"
)

# 70% training data
summary_train = create_deviation_summary(
    X_train,
    y_train,
    "70% Used in Training"
)

# 30% test data
summary_test = create_deviation_summary(
    X_test,
    y_test,
    "30% Not Used in Training"
)



100% of Data


,Deviation Buckets,# Projects,% of Projects,Actual Human,Model Estimate Median
0,<1 hour,15840,34%,5.85,4.29
1,1-2 hours,8581,19%,7.98,6.12
2,2-3 hours,5410,12%,9.89,7.69
3,3-7 hours,8920,19%,13.13,10.31
4,>7 hours,7173,16%,34.38,17.06



70% Used in Training


,Deviation Buckets,# Projects,% of Projects,Actual Human,Model Estimate Median
0,<1 hour,12254,38%,6.03,4.45
1,1-2 hours,6375,20%,8.31,6.37
2,2-3 hours,3796,12%,10.46,8.23
3,3-7 hours,5699,18%,14.65,11.49
4,>7 hours,4022,13%,39.10,18.83



30% Not Used in Training


,Deviation Buckets,# Projects,% of Projects,Actual Human,Model Estimate Median
0,<1 hour,3586,26%,5.25,3.87
1,1-2 hours,2206,16%,7.03,5.45
2,2-3 hours,1614,12%,8.55,6.61
3,3-7 hours,3221,23%,10.42,8.64
4,>7 hours,3151,23%,28.35,15.40


In [9]:
# =====================================
# FEATURE IMPORTANCE FOR BEST MODEL
# =====================================

rf_model = models["Random Forest"]

feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "Importance",
    ascending=False
)

feature_importance.head(20)


,Feature,Importance
2,Test Count,0.152506
484,Service Catalog Segment_TST,0.076165
515,Service Catalog Sub Category_Global Market Ser...,0.058565
595,Has Test Task Flag_Yes,0.034158
349,Service Program_New Construction,0.026640
0,Standard Count,0.019593
4,Confirmation Year_2026,0.018064
5,Handler Region_Americas,0.016064
543,CCN_AAAE,0.016032
589,CCN_UNKNOWN,0.015901


In [10]:
# =====================================
# FEATURES USED BY FINAL MODEL
# =====================================

rf_model = models["Random Forest"]

print("Number of Features Used:", len(X_train.columns))
print("\nFeatures Used:\n")

for i, col in enumerate(X_train.columns, start=1):
    print(f"{i}. {col}")


Number of Features Used: 695

Features Used:

1. Standard Count
2. Flex Project Count
3. Test Count
4. Confirmation Year_2025, 2026
5. Confirmation Year_2026
6. Handler Region_Americas
7. Handler Region_EMEA
8. Product Group_Audio/Video Equipment
9. Product Group_Automotive
10. Product Group_Battery Chargers
11. Product Group_CSP
12. Product Group_Cabinets
13. Product Group_Cameras
14. Product Group_Circuit Breakers
15. Product Group_Computers
16. Product Group_Computing & Peripherals
17. Product Group_Controls
18. Product Group_DG Inverters
19. Product Group_Display & Monitors
20. Product Group_Display Devices
21. Product Group_Displays and Monitors
22. Product Group_Electric Fans - Portable Household Fans
23. Product Group_Electrically Operated Toys
24. Product Group_Entertainment Systems
25. Product Group_Fans
26. Product Group_Gaming Equipment
27. Product Group_Household Cooking Appliances - Cooking Appliance
28. Product Group_Household Cooking Appliances - Hot Beverage Appliance
2

In [11]:
# =====================================
# ROLL FEATURE IMPORTANCE BACK TO ORIGINAL BUSINESS COLUMNS
# =====================================

rf_model = models["Random Forest"]

encoded_importance = pd.DataFrame({
    "Encoded Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

original_cols = X_controlled.columns.tolist()

def map_to_original_col(encoded_feature):
    for col in sorted(original_cols, key=len, reverse=True):
        if encoded_feature == col or encoded_feature.startswith(col + "_"):
            return col
    return encoded_feature

encoded_importance["Original Business Column"] = encoded_importance["Encoded Feature"].apply(map_to_original_col)

business_importance = (
    encoded_importance
    .groupby("Original Business Column", as_index=False)["Importance"]
    .sum()
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(business_importance.head(15))


,Original Business Column,Importance
0,Test Count,0.152506
1,Service Catalog Sub Category,0.119026
2,CCN,0.090556
3,Product Type,0.083097
4,Service Catalog Segment,0.079483
5,Service Program,0.072219
6,Product Group,0.059220
7,Ship to Account Number,0.051871
8,Service Detail,0.036859
9,Service Catalog Item Number,0.035076


In [16]:
# =====================================
# TOP 10 FEATURE MODEL TEST
# =====================================

top_features = [
    "Ship to Account Number",
    "Test Count",
    "Service Catalog Sub Category",
    "CCN",
    "Product Type",
    "Service Catalog Segment",
    "Service Program",
    "Product Group",
    "Service Catalog Item Number",
    "Service Detail"
]

# Create new X using only top features
X_top = model_data[top_features].copy()
y_top = model_data["Log_Eng_Hours"].copy()

 # Treat account number as a category, not a true number
if "Ship to Account Number" in X.columns:
    X_top["Ship to Account Number"] = X_top["Ship to Account Number"].astype("string")

# # print(X_top["Ship to Account Number"].nunique())

# Find categorical columns
cat_cols_top = X_top.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

# Same controlled encoding logic
for col in cat_cols_top:

    X_top[col] = (
        X_top[col]
        .astype("string")
        .fillna("UNKNOWN")
        .str.strip()
    )

    top_categories = (
        X_top[col]
        .value_counts()
        .head(50)
        .index
    )

    X_top[col] = np.where(
        X_top[col].isin(top_categories),
        X_top[col],
        "OTHER"
    )

# One-hot encode
X_top_encoded = pd.get_dummies(
    X_top,
    columns=cat_cols_top,
    drop_first=True,
    dtype=int
)

print("Top 10 model shape:", X_top_encoded.shape)

# Train/test split
X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(
    X_top_encoded,
    y_top,
    test_size=0.30,
    random_state=42
)

# Train RF
rf_top = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

rf_top.fit(X_train_top, y_train_top)

# Predict
y_pred_log = rf_top.predict(X_test_top)

# Convert back to hours
y_true_hours = np.expm1(y_test_top)
y_pred_hours = np.expm1(y_pred_log)

# MAE
mae_top = mean_absolute_error(
    y_true_hours,
    y_pred_hours
)

print("\nTop 10 Feature Model MAE:", round(mae_top, 2))


Top 10 model shape: (45924, 418)

Top 10 Feature Model MAE: 6.55


In [17]:
# ==========================================
# TOP 10 MODEL
# 100% / 70% / 30% BUCKET ANALYSIS
# ==========================================

import pandas as pd
import numpy as np

def create_top10_bucket_summary(X_data, y_data, label):

    # Predictions
    pred_log = rf_top.predict(X_data)

    # Convert back to hours
    actual_hours = np.expm1(y_data)
    predicted_hours = np.expm1(pred_log)

    # Results table
    results = pd.DataFrame({
        "Actual Hours": actual_hours,
        "Predicted Hours": predicted_hours
    })

    # Absolute error
    results["Absolute Error"] = abs(
        results["Actual Hours"] -
        results["Predicted Hours"]
    )

    # Buckets
    results["Error Bucket"] = pd.cut(
        results["Absolute Error"],
        bins=[0,1,2,3,7,np.inf],
        labels=[
            "<1 hour",
            "1-2 hours",
            "2-3 hours",
            "3-7 hours",
            ">7 hours"
        ],
        include_lowest=True
    )

    # Summary table
    summary = (
        results
        .groupby("Error Bucket", observed=False)
        .agg(
            Count=("Absolute Error","count"),
            Percent=("Absolute Error",
                     lambda x: round(
                         len(x)/len(results)*100,
                         2
                     )),
            Actual_Hours_Median=("Actual Hours","median"),
            Predicted_Hours_Median=("Predicted Hours","median"),
            Median_Error=("Absolute Error","median")
        )
        .reset_index()
    )

    print("\n" + "="*60)
    print(label)
    print("="*60)

    display(summary)

    # Quick summary
    within_1 = results["Absolute Error"].le(1).mean()*100
    within_2 = results["Absolute Error"].le(2).mean()*100
    within_3 = results["Absolute Error"].le(3).mean()*100
    within_7 = results["Absolute Error"].le(7).mean()*100

    print("\nAccuracy Summary:")
    print(f"Within 1 hour: {within_1:.2f}%")
    print(f"Within 2 hours: {within_2:.2f}%")
    print(f"Within 3 hours: {within_3:.2f}%")
    print(f"Within 7 hours: {within_7:.2f}%")

    return summary


# ==========================================
# 100% OF DATA
# ==========================================

summary_top_100 = create_top10_bucket_summary(
    X_top_encoded,
    y_top,
    "100% OF DATA"
)

# ==========================================
# 70% TRAINING DATA
# ==========================================

summary_top_train = create_top10_bucket_summary(
    X_train_top,
    y_train_top,
    "70% USED IN TRAINING"
)

# ==========================================
# 30% TEST DATA
# ==========================================

summary_top_test = create_top10_bucket_summary(
    X_test_top,
    y_test_top,
    "30% NOT USED IN TRAINING"
)



100% OF DATA


,Error Bucket,Count,Percent,Actual_Hours_Median,Predicted_Hours_Median,Median_Error
0,<1 hour,13210,28.76,4.20,4.320341,0.441963
1,1-2 hours,8510,18.53,5.50,5.657825,1.435618
2,2-3 hours,5588,12.17,7.25,7.219084,2.464930
3,3-7 hours,10185,22.18,9.75,9.343114,4.428109
4,>7 hours,8431,18.36,24.00,14.963530,12.542973



Accuracy Summary:
Within 1 hour: 28.76%
Within 2 hours: 47.30%
Within 3 hours: 59.46%
Within 7 hours: 81.64%

70% USED IN TRAINING


,Error Bucket,Count,Percent,Actual_Hours_Median,Predicted_Hours_Median,Median_Error
0,<1 hour,9979,31.04,4.25,4.471792,0.441963
1,1-2 hours,6209,19.32,5.50,5.745678,1.430829
2,2-3 hours,3965,12.33,7.50,7.432753,2.464075
3,3-7 hours,6846,21.30,10.50,9.894467,4.373843
4,>7 hours,5147,16.01,25.75,15.244663,12.140433



Accuracy Summary:
Within 1 hour: 31.04%
Within 2 hours: 50.36%
Within 3 hours: 62.69%
Within 7 hours: 83.99%

30% NOT USED IN TRAINING


,Error Bucket,Count,Percent,Actual_Hours_Median,Predicted_Hours_Median,Median_Error
0,<1 hour,3231,23.45,4.00,3.949181,0.443306
1,1-2 hours,2301,16.70,5.10,5.398288,1.454897
2,2-3 hours,1623,11.78,6.75,6.586535,2.466826
3,3-7 hours,3339,24.23,8.50,8.543837,4.519571
4,>7 hours,3284,23.84,21.00,14.415968,13.351289



Accuracy Summary:
Within 1 hour: 23.45%
Within 2 hours: 40.15%
Within 3 hours: 51.93%
Within 7 hours: 76.16%
